# Lab 03A: Reflexion — Self-Improvement with Verbal Memory

**Week 3 — Agentic AI: Building Autonomous Intelligent Systems**

Reflection lets an agent critique its own work. **Reflexion** goes one step further: after each attempt the agent writes itself a short **verbal lesson** and stores it as **memory**, so the *next* attempt starts smarter. That growing memory of "what went wrong and what to do differently" is the whole idea — and your first taste of an agent that carries **state** across attempts.

You will build the loop from scratch with the Gemini API, then wrap it in a small **Gradio** app so you can type a task and watch the agent improve.

## Introduction

This lesson answers:

- What is Reflexion, and how is it different from a plain reflection loop?
- Where does memory come in, and why does verbal feedback help the next attempt?
- What building blocks does a Reflexion loop need?
- What makes a self-critiquing loop trustworthy (and what makes it degrade)?

## Learning Goals

After completing this lesson you will be able to:

- Explain the Reflexion loop: **attempt -> evaluate -> reflect (store memory) -> retry**.
- Use a *separate* evaluator with a structured rubric to score a draft.
- Turn a critique into a short verbal lesson and feed it back as memory.
- Add a principled stopping rule (bar met / score threshold / hard cap).
- Wrap the loop in a **Gradio** UI.

You will implement `attempt`, `evaluate`, `reflect`, and `reflexion_loop`, then a Gradio app around them. This notebook runs in **Google Colab** on the **Gemini API**.

## What is Reflexion?

A plain reflection loop is *generate -> critique -> revise*. **Reflexion** adds **memory**: instead of just revising, the agent writes itself a short **verbal lesson** ("Next attempt: keep it under 8 words") and stores it. Every future attempt reads all the accumulated lessons first. The agent isn't just editing one draft — it's learning across attempts, carrying that learning as state.

```
   task
     │
     ▼
   ┌────────────┐   draft
   │  ATTEMPT   │───────────────┐   reads the accumulated
   │ (writer)   │◀──────────────┤   reflections each time
   └────────────┘   reflections │
     │                          │
     ▼                          │
   ┌────────────┐  score,       │
   │  EVALUATE  │  issues       │
   │ (judge)    │               │
   └────────────┘               │
     │                          │
     ├── good enough? ──yes──▶ final answer
     │ no                       │
     ▼                          │
   ┌────────────┐  "Next attempt: ..."  (one verbal lesson)
   │  REFLECT   │───────────────┘
   │ (coach)    │   store it as MEMORY
   └────────────┘
```

Three distinct roles — writer, judge, coach — and one growing list of lessons that ties them together.

## Use cases

Reflexion helps whenever a first attempt is often wrong but *checkable*, so feedback can steer the next try:

- **Constraint satisfaction** — writing that must obey rules (length, banned words, format).
- **Code that must pass tests** — run the tests, feed failures back as lessons.
- **Reasoning with a verifier** — math/logic where an evaluator can catch mistakes.
- **Tool-using tasks with an environment** — retry after an action fails, remembering *why* it failed.
- **Anything with a rubric** — the evaluator scores, the reflection says how to score higher.

## Building blocks

- **An attempter** — produces a draft, and reads the accumulated reflections first.
- **An evaluator** — a *separate* critic that scores the draft against a rubric (structured output).
- **A reflector** — turns the critique into one short verbal lesson to remember.
- **Memory** — the list of lessons, injected into every future attempt (this is the agent's state).
- **A stopping rule** — stop when the bar is met, a score threshold is cleared, or a hard iteration cap is hit.

## Considerations for trustworthy self-critique

- **Separate the critic from the author.** A model grading its own fresh output tends to be generous; a distinct evaluator persona + a concrete rubric grades harder.
- **Always cap the loop.** A hard `max_iters` guarantees termination even if the score never clears the bar.
- **Watch for degradation.** More iterations can make output *worse* — keep the best draft, not just the last.
- **Keep reflections concrete.** "Be better" is useless; "shorten to 8 words and drop the banned term" is actionable.
- **Persistent memory can go stale.** Because the memory file accumulates across runs, old lessons can pull new attempts in the wrong direction; call `memory.reset()` to start clean, and cap how many lessons `recall()` injects (prompts grow with memory).

## Setup: add your Gemini API key as a Colab secret

1. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey).
2. In Colab, click the **key icon** in the left sidebar ("Secrets").
3. Add a new secret named **`GEMINI_API_KEY`** and paste your key as the value.
4. Toggle **"Notebook access"** on for that secret.

The next cell installs the SDK plus **Gradio** (for the UI at the end).

In [ ]:
!pip install -q google-genai python-dotenv gradio

In [ ]:
import time

from google.colab import userdata
from dotenv import load_dotenv
from google import genai
from google.genai import errors
from pydantic import BaseModel

load_dotenv()

api_key = userdata.get('GEMINI_API_KEY')
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")

client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash-lite"
JUDGE_MODEL = "gemini-3.1-flash-lite"

# Quick connectivity check
print(client.models.generate_content(model=MODEL, contents="Say 'Setup complete!' and nothing else.").text)

Setup complete!


## Helpers: text and structured JSON

Two helpers, both routed through `_call`, which retries the transient errors a multi-call loop hits back to back: a **429** rate limit (wait 60s) and a **503** "model overloaded" (wait 10s).

- `generate()` — plain text back (a draft, a lesson).
- `generate_json(prompt, schema)` — schema-validated JSON; `response.parsed` returns a typed object.

In [ ]:
def _call(model: str = MODEL,**kwargs):
    """Call Gemini, retrying on transient 429 (rate limit) and 503 (overloaded) errors."""
    for attempt in range(4):  # the first call + up to 3 retries
        try:
            return client.models.generate_content(model=model, **kwargs)
        except errors.ClientError as e:
            if e.code == 429 and attempt < 3:
                print("  Rate limited (429). Waiting 60s, then retrying...")
                time.sleep(60)
            else:
                raise
        except errors.ServerError as e:
            if e.code == 503 and attempt < 3:
                print("  Model overloaded (503). Waiting 10s, then retrying...")
                time.sleep(10)
            else:
                raise


def generate(prompt: str,model: str = MODEL) -> str:
    """Send one prompt to Gemini and return the response text."""
    return (_call(model=model,contents=prompt).text or "").strip()

def generate_json(prompt: str, schema,model: str = MODEL):
    """Generate schema-validated JSON. `schema` is a pydantic model or typed list."""

    response = _call(
        contents=prompt,
        model=model,
        config={"response_mime_type": "application/json", "response_schema": schema},
    )
    return response.parsed

## The task (hard enough that the first try usually needs work)

Reflexion is only interesting when the first attempt is *not* already perfect — otherwise there is nothing to remember. So we pick a task with a genuinely tricky constraint: **write a slogan that never uses the letter "e"** (plus a word-count and length bound). Avoiding the most common letter in English while staying catchy is hard, so the loop almost always needs a few rounds — and the lessons it stores ("the word 'refill' has an e") are wonderfully concrete.

In [ ]:
SAMPLE_TASK = (
    "Write a short, catchy slogan for a reusable water bottle brand called 'Loop'.\n"
    "Hard constraints:\n"
    "- Use exactly 6 words.\n"
    "- Do NOT use the letter 'e' anywhere in the slogan.\n"
    "- No word longer than 8 letters.\n"
    "- It should read like a real brand slogan.\n"
    "- It should have 'Loop' in the slogan and a human expression."
)
print(SAMPLE_TASK)

Write a short, catchy slogan for a reusable water bottle brand called 'Loop'.
Hard constraints:
- Use exactly 6 words.
- Do NOT use the letter 'e' anywhere in the slogan.
- No word longer than 8 letters.
- It should read like a real brand slogan.
- It should have 'Loop' in the slogan and a human expression.


## The evaluator (deterministic checks + the LLM)

The hard constraints (no "e", word count, word length) are **objective**, so we check them in plain Python — the model can't grade its own letter-counting generously. The LLM only judges the *subjective* part (is it catchy, does it read like a slogan). Combining a deterministic checker with an LLM judge is a pattern you will reuse whenever some criteria are checkable and some are a matter of taste.

`Critique` is a `pydantic` model; `meets` is true only when there are **zero** objective violations *and* the LLM says it reads like a real slogan.

In [ ]:
class Critique(BaseModel):
    score: int            # 1-10, 10 = flawless
    meets: bool           # true only if EVERY constraint holds
    issues: list[str]     # specific problems
    fixes: list[str]      # one concrete fix per issue


class Style(BaseModel):
    catchy: int              # 1-10, subjective
    reads_like_slogan: bool


def check_constraints(draft: str) -> list[str]:
    """Deterministic checks for the OBJECTIVE constraints -- no LLM, so nothing is missed."""
    words = [w.strip(".,!?;:") for w in draft.split()]
    violations = []
    if "e" in draft.lower():
        offenders = [w for w in draft.split() if "e" in w.lower()]
        violations.append("uses the letter 'e' in: " + ", ".join(offenders))
    if len(words) != 6:
        violations.append(f"word count is {len(words)} (must be exactly 6)")
    too_long = [w for w in words if len(w) > 8]
    if too_long:
        violations.append("word(s) longer than 8 letters: " + ", ".join(too_long))
    return violations


def evaluate(task: str, draft: str) -> Critique:
    """Hybrid evaluator: deterministic checks for the hard constraints + the LLM for style."""
    # 1) Objective constraints -- checked in code, so they are always caught.
    violations = check_constraints(draft)

    # 2) Subjective judgment -- only the LLM can say whether it is catchy.

    style = generate_json(
        f"""# Role
You are a senior brand strategist judging slogans on STYLE only. You are sharp, selective and confident Brand Lead who knows what sells fast.

# Task
Rate how catchy and slogan-like the draft is (ignore the hard constraints; those are checked separately) which is more memorable, more selling.
Be very picky about the best slogan and make the scoring difficult.

# Format
JSON with keys catchy (int, 1-10) and reads_like_slogan (bool).

# Input
{draft}""",
        schema=Style,
        model=JUDGE_MODEL,
    )

    meets = (not violations) and style.reads_like_slogan
    # Objective violations cap the score low; otherwise use the style score, clamped to 1-10.
    score = 4 if violations else max(1, min(10, style.catchy))
    issues = violations + ([] if style.reads_like_slogan else ["does not read like a real slogan"])
    fixes = [f"Remove/rework: {v}" for v in violations]
    return Critique(score=score, meets=meets, issues=issues, fixes=fixes)

## The memory: a Markdown file on disk

Here is the concrete answer to "where does the memory live?" — in a **Markdown file**, `/content/reflexion_memory.md`. It is a human-readable "lessons learned" log you can open in the Colab file browser: a title line and one `- ` bullet per lesson.

Because it is a real file, the memory **persists**: it survives a kernel restart and **accumulates across runs** (every run appends to it). It also starts **pre-seeded** with one lesson, so the very first attempt already has something to recall.

The `ReflexionMemory` object below is just a thin wrapper over that file with two named doors:

- **`remember(lesson)`** — appends a lesson and re-saves the file (watch for `[MEMORY] stored ...`).
- **`recall(last_n)`** — reads the most recent lessons back for the next prompt.

```
# Reflexion memory

- Next attempt: count the words (need exactly 6) and scan every word for the letter 'e' before finalizing.
- Next attempt: replace a clunky literal phrase with a punchier brand statement.
```

In [ ]:
import os

MEMORY_PATH = "/content/reflexion_memory.md"

# The memory starts pre-seeded with at least one lesson, so attempt 1 already has
# something to recall. On a fresh Colab the file is created from this seed.
SEED_LESSONS = [
    "Next attempt: count the words (need exactly 6) and scan every word for the letter 'e' before finalizing.",
]


class ReflexionMemory:
    """The agent's memory, stored in a Markdown file on disk.

    The file (MEMORY_PATH) IS the memory. It persists across runs and kernel restarts,
    and each run appends to it. `self.lessons` is just the in-RAM copy loaded from the file.
    """

    def __init__(self, path: str = MEMORY_PATH):
        self.path = path
        self.lessons: list[str] = self._load()   # <-- read the memory from disk (seeding if new)

    def _load(self) -> list[str]:
        """Read lessons from the Markdown file; if it does not exist yet, write the seed."""
        if not os.path.exists(self.path):
            self.lessons = list(SEED_LESSONS)
            self._save()
            return list(SEED_LESSONS)
        lessons = []
        for line in open(self.path):
            line = line.rstrip("\n")
            if line.startswith("- "):        # each lesson is one Markdown bullet
                lessons.append(line[2:])
        return lessons

    def _save(self) -> None:
        """Rewrite the whole Markdown file: a title, then one bullet per lesson."""
        with open(self.path, "w") as f:
            f.write("# Reflexion memory\n\n")
            for lesson in self.lessons:
                f.write(f"- {lesson}\n")

    def remember(self, lesson: str) -> None:
        """WRITE one lesson into memory (and persist it to the file)."""
        self.lessons.append(lesson)
        self._save()
        print(f"  [MEMORY] stored lesson #{len(self.lessons)} (saved to {self.path}): {lesson}")

    def recall(self, last_n: int = 5) -> str:
        """READ the most recent lessons back, formatted for the next prompt."""
        recent = self.lessons[-last_n:]
        if not recent:
            return "(memory is empty)"
        return "\n".join(f"- {lesson}" for lesson in recent)

    def reset(self) -> None:
        """Wipe the accumulated memory back to just the seed lesson(s)."""
        self.lessons = list(SEED_LESSONS)
        self._save()

    def __len__(self) -> int:
        return len(self.lessons)

## The attempter (reads its memory first)

`attempt` starts by **reading** the stored lessons with `memory.recall()` and folding them into the prompt — so every draft is shaped by everything remembered so far. On the first pass memory is empty; by later passes it is working against concrete reminders it wrote for itself.

In [ ]:
def attempt(task: str, memory: "ReflexionMemory") -> str:
    """Produce a draft, applying the lessons READ from memory (memory.recall())."""
    return generate(
        f"""# Role
You are a senior brand copywriter.

# Context
You may have attempted this task before. Apply the lessons you wrote for yourself.

# Task
Complete the task in the Input section, satisfying EVERY constraint.

# Lessons from earlier attempts (read from memory)
{memory.recall()}

# Format
Return only the final line of copy -- no preamble, no quotes.

# Input
{task}"""
    )

## The reflector (turns a critique into one lesson)

This is the step that makes it *Reflexion* rather than plain reflection: instead of editing the draft, the coach distills the critique into a single, concrete lesson — a sentence the next attempt can act on. That sentence is what gets stored in memory.

In [ ]:
def reflect(task: str, draft: str, critique: Critique) -> str:
    """Turn a critique into ONE short verbal lesson to remember for next time."""
    issues = "\n".join(f"- {i}" for i in critique.issues) or "(none listed)"
    return generate(
        f"""# Role
You are a senior writing coach.

# Task
Write ONE short lesson -- a single sentence starting with "Next attempt:" -- that will help the next try fix the problems below. Be concrete and specific; name the exact change to make.

# Input
## Task
{task}

## Draft that fell short
{draft}

## Issues found
{issues}"""
    )

## The Reflexion loop

Each round: read memory -> **attempt** -> **evaluate** -> stop if the bar is met, otherwise **reflect** and **write the lesson to memory**. The `memory` object is created once at the top of the run and threaded through every attempt. Two guards keep the demo honest and finite: `min_iters` (always run at least a couple of rounds, so the memory is visible) and `max_iters` (a hard cap). Together they pin the run to **2-4 iterations**.

In [ ]:
def reflexion_loop(task: str, threshold: int = 9, min_iters: int = 2, max_iters: int = 4) -> dict:
    """Read memory -> attempt -> evaluate -> reflect(WRITE to memory) -> retry.

    Stops when the draft clears the bar AND at least `min_iters` rounds have run, or when
    `max_iters` is hit. Returns the final draft, a per-iteration history, and the memory.
    """
    memory = ReflexionMemory()   # <-- the memory store for this run lives right here
    history = []
    draft = ""

    best_draft=""
    best_score=0

    prev_crit_issue=None
    for i in range(1, max_iters + 1):
        print(f"[Iter {i}] reading {len(memory)} lesson(s) from memory...")
        draft = attempt(task, memory)          # attempt READS memory (memory.recall)
        crit = evaluate(task, draft)
        history.append({"iter": i, "draft": draft, "score": crit.score,
                        "meets": crit.meets, "issues": crit.issues})

        print(f"[Iter {i}] score={crit.score}/10  meets={crit.meets}  draft: {draft}")

        if crit.score > best_score:
          best_score=crit.score
          best_draft=draft

        new_crit_issue=sorted(crit.issues)
        if prev_crit_issue==new_crit_issue:
          break
        else:
          prev_crit_issue=new_crit_issue

        cleared = crit.meets or crit.score >= threshold
        if cleared and i >= min_iters:
            print(f"  Bar cleared at iteration {i}.")
            break

        if i < max_iters:   # only store a lesson if there will be another attempt to use it
            if cleared:
                lesson = "Next attempt: constraints already hold -- keep them and make it punchier."
            else:
                lesson = reflect(task, draft, crit)
            memory.remember(lesson)            # <-- WRITE the lesson to memory

    return {"final": best_draft, "history": history, "memory": memory.lessons}

## Run it

Watch the score climb (usually) as the reflective memory fills up. The final block prints the lessons the agent wrote for itself — the memory that made the later attempts better.

In [ ]:
result = reflexion_loop(SAMPLE_TASK, threshold=9, min_iters=2, max_iters=4)

print("\nFinal answer:", result["final"])
print("\nWhat ended up in memory (result['memory']):")
for lesson in result["memory"]:
    print(" -", lesson)
if not result["memory"]:
    print("  (empty)")

[Iter 1] reading 12 lesson(s) from memory...
[Iter 1] score=3/10  meets=False  draft: Loop your thirst, slak your bliss.
  [MEMORY] stored lesson #13 (saved to /content/reflexion_memory.md): Next attempt: Swap "slak" for "quench" to make the human expression more natural and less awkward.
[Iter 2] reading 13 lesson(s) from memory...
[Iter 2] score=6/10  meets=True  draft: Loop your thirst, find your bliss.
  Bar cleared at iteration 2.

Final answer: Loop your thirst, find your bliss.

What ended up in memory (result['memory']):
 - Next attempt: count the words (need exactly 6) and scan every word for the letter 'e' before finalizing.
 - Next attempt: Replace "good, for, all" with a two-word phrase that conveys a positive human feeling or action related to drinking, ensuring it follows all other constraints.
 - Next attempt: add one more word to your current slogan to reach the six-word count and consider a word that implies a positive, human feeling beyond just "happy."
 - Next attemp

## Look at the memory file on disk

The memory is a real file. Print it to see what the agent has written to `/content/reflexion_memory.md` — the seed lesson plus everything it has learned across runs. (Open it from the Colab file browser too, or run this cell again after another loop to watch it grow.)

In [ ]:
with open(MEMORY_PATH) as f:
    print(f.read())

# Reflexion memory

- Next attempt: count the words (need exactly 6) and scan every word for the letter 'e' before finalizing.
- Next attempt: Replace "good, for, all" with a two-word phrase that conveys a positive human feeling or action related to drinking, ensuring it follows all other constraints.
- Next attempt: add one more word to your current slogan to reach the six-word count and consider a word that implies a positive, human feeling beyond just "happy."
- Next attempt: add one more word to "Loop your thirst, find joy" that fits the constraints and expresses human enjoyment.
- Next attempt: add one more word to the slogan, making sure it still follows all other constraints.
- Next attempt: Remove the word "Quench" and replace it with a six-letter word that has no "e" and expresses the idea of drinking, then add a human expression.
- Next attempt: Add one more word to the slogan to reach the required six-word count.
- Next attempt: swap "quench" for a synonym without the letter

## A Gradio UI

`gradio` turns a Python function into a web UI in a few lines. We wrap `reflexion_loop` so you can type any task, set the threshold and iteration cap, and see the final answer, the per-iteration scores, and the reflective memory.

`demo.launch()` starts the app; in Colab it prints a **public share URL** you can open in a new tab. (If a cell is still running the app, click the stop button before re-launching.)

In [ ]:
import gradio as gr


def run_reflexion_ui(task: str, threshold: int, min_iters: int, max_iters: int):
    """Run the loop and return three text blocks for the UI."""
    result = reflexion_loop(task, threshold=int(threshold),
                            min_iters=int(min_iters), max_iters=int(max_iters))
    scores = "\n".join(
        f"Iter {h['iter']}: score {h['score']}/10 (meets={h['meets']})"
        for h in result["history"]
    )
    memory = "\n".join(f"- {lesson}" for lesson in result["memory"]) or "(empty)"
    #return result["final"], scores, memory

    print("History",result['history'])
    steps = []
    for hist in result["history"]:
        status = "PASS" if hist["meets"] else "FAIL"
        step_text = (
            f"--- ATTEMPT {hist['iter']} ---\n"
            f"Draft: \"{hist['draft']}\"\n"
            f"Score: {hist['score']}/10 ({status})\n"
            f"Issues: {', '.join(hist['issues']) if hist['issues'] else 'None'}\n"
        )
        steps.append(step_text)

    # Join everything into one big text block separated by spacing
    history_text = "\n".join(steps)
    memory = "\n".join(f"- {lesson}" for lesson in result["memory"]) or "(empty)"

    return result["final"], history_text, memory

demo = gr.Interface(
    fn=run_reflexion_ui,
    inputs=[
        gr.Textbox(label="Task", lines=7, value=SAMPLE_TASK),
        gr.Slider(1, 10, value=9, step=1, label="Score threshold"),
        gr.Slider(1, 4, value=2, step=1, label="Minimum iterations"),
        gr.Slider(1, 6, value=4, step=1, label="Max iterations"),
    ],
    outputs=[
        gr.Textbox(label="Final answer"),
        gr.Textbox(label="Iteration scores"),
        gr.Textbox(label="Reflective memory (result['memory'])"),
    ],
    title="Reflexion: self-improving with verbal memory",
    description="The agent attempts the task, a separate judge scores it, a coach stores a lesson in memory, and it retries.",
)

demo.launch()  # in Colab this prints a public share URL

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8c066b53a5d07f736b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Your turn (exercises)

1. **Keep the best, not the last.** The loop returns the final draft even if an earlier iteration scored higher. Track the best-scoring draft and return that instead.
2. **Tougher rubric.** Add a fifth constraint to the task and confirm the evaluator catches violations and the reflections adapt.
3. **Convergence stop.** Stop early if two consecutive iterations produce the same `issues` (the reflection isn't helping), not only on the score.
4. **A truly independent critic.** Point `evaluate` at a different model, or give it a much stricter system prompt, so the judge is clearly separate from the author.
5. **Richer UI.** Extend the Gradio app to show each attempt's draft alongside its score, so you can watch the improvement step by step.

When you're done, save a copy (**File -> Save a copy in Drive**) and submit your notebook link via Canvas.